In [ ]:
import re, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)
sns.set_theme(style="whitegrid")
print("Libraries imported successfully.")

In [ ]:
LOCAL_FILE = Path("factory_production_dataset.xlsx")
GITHUB_RAW_URL = "https://raw.githubusercontent.com/GeethaGunasekaran1/Dataset_rep/main/factory_production_dataset.xlsx"

if LOCAL_FILE.exists():
    df = pd.read_excel(LOCAL_FILE)
    print("Loaded local workbook:", LOCAL_FILE)
else:
    try:
        df = pd.read_excel(GITHUB_RAW_URL)
        print("Loaded workbook directly from GitHub.")
    except Exception as exc:
        raise FileNotFoundError(
            "Could not load the GitHub workbook automatically. Download "
            "'factory_production_dataset.xlsx' from the repository and place it "
            "beside this notebook, then run this cell again. Original error: " + str(exc)
        )

# Clean header formatting
df.columns = [re.sub(r"[^a-z0-9]+", "_", str(c).strip().lower()).strip("_") for c in df.columns]
print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head())

In [ ]:
print("Dataset information:")
df.info()
print("\nData types:")
display(df.dtypes.to_frame("data_type"))
print("\nFirst five rows:")
display(df.head())
print("\nLast five rows:")
display(df.tail())
print("\nSummary statistics:")
display(df.describe(include="all").T)

In [ ]:
# Convert columns whose names suggest dates/timestamps
for col in df.columns:
    if any(word in col for word in ["date", "time", "timestamp"]):
        converted = pd.to_datetime(df[col], errors="coerce")
        # Keep the conversion only if at least one valid date was parsed
        if converted.notna().any():
            df[col] = converted

# Convert object columns to numeric only when nearly all non-empty values parse as numbers
for col in df.select_dtypes(include="object").columns:
    parsed = pd.to_numeric(df[col].astype(str).str.replace(",", "", regex=False), errors="coerce")
    non_empty = df[col].notna().sum()
    if non_empty and parsed.notna().sum() / non_empty >= 0.90:
        df[col] = parsed

numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
categorical_cols = df.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
date_cols = df.select_dtypes(include=["datetime", "datetimetz"]).columns.tolist()
print("Numeric columns:", numeric_cols)
print("Categorical columns:", categorical_cols)
print("Date columns:", date_cols)

if numeric_cols:
    arr = df[numeric_cols].to_numpy(dtype=float)
    print("NumPy numeric array shape:", arr.shape)
    print("First five rows of NumPy array:")
    print(arr[:5])
    print("Column-wise means (ignoring NaN):", np.nanmean(arr, axis=0))

In [ ]:
print("Missing values detected with isnull():")
display(df.isnull().sum().to_frame("missing_count"))
print("Missing values detected with isna():")
display(df.isna().sum().to_frame("missing_count"))
print("Duplicate rows:", df.duplicated().sum())

df = df.drop_duplicates().copy()

# Recalculate field groups after duplicate removal
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
categorical_cols = df.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
date_cols = df.select_dtypes(include=["datetime", "datetimetz"]).columns.tolist()

# Fill numeric missing values with median; categorical with mode
for col in numeric_cols:
    med = df[col].median()
    if pd.notna(med):
        df[col] = df[col].fillna(med)

for col in categorical_cols:
    modes = df[col].mode(dropna=True)
    if not modes.empty:
        df[col] = df[col].fillna(modes.iloc[0])

# Fill missing date values in chronological order, if date columns exist
for col in date_cols:
    df = df.sort_values(col)
    df[col] = df[col].ffill().bfill()

# Normalize text spacing and casing for category-like text columns
for col in categorical_cols:
    df[col] = df[col].astype("string").str.strip().str.replace(r"\s+", " ", regex=True)

# Flag negative values in columns that represent non-negative production quantities/costs/durations
for col in numeric_cols:
    name = col.lower()
    if any(term in name for term in ["production", "produced", "output", "quantity", "units", "defect", "waste", "downtime", "duration", "cost", "energy", "consumption", "sales", "revenue"]):
        negative_count = (df[col] < 0).sum()
        if negative_count:
            print(f"{col}: found {negative_count} negative values; replacing with NaN and median.")
            df.loc[df[col] < 0, col] = np.nan
            med = df[col].median()
            if pd.notna(med):
                df[col] = df[col].fillna(med)

print("Shape after removing duplicates:", df.shape)
print("Remaining missing values:")
display(df.isna().sum().to_frame("missing_after_cleaning"))
display(df.head())

In [ ]:
def find_col(candidates):
    for candidate in candidates:
        if candidate in df.columns:
            return candidate
    # allow partial-name matching
    for candidate in candidates:
        matches = [c for c in df.columns if candidate in c]
        if matches:
            return matches[0]
    return None

production_col = find_col(["production_quantity", "units_produced", "total_production", "production", "output_quantity", "output", "quantity_produced", "units"])
defect_col = find_col(["defects", "defect_count", "defective_units", "rejected_units"])
downtime_col = find_col(["downtime", "downtime_hours", "machine_downtime"])
target_col = find_col(["target", "production_target", "target_quantity", "planned_production"])
machine_col = find_col(["machine", "machine_id", "machine_name", "equipment"])
product_col = find_col(["product", "product_id", "product_name"])
shift_col = find_col(["shift", "work_shift"])
date_col = date_cols[0] if date_cols else None

print("Detected fields:")
for label, value in {
    "production":production_col, "defects":defect_col, "downtime":downtime_col,
    "target":target_col, "machine":machine_col, "product":product_col,
    "shift":shift_col, "date":date_col
}.items():
    print(f"{label}: {value}")

if production_col and defect_col:
    denominator = df[production_col].replace(0, np.nan)
    df["defect_rate_pct"] = (df[defect_col] / denominator) * 100

if production_col and target_col:
    denominator = df[target_col].replace(0, np.nan)
    df["target_achievement_pct"] = (df[production_col] / denominator) * 100

if production_col and downtime_col:
    df["production_per_downtime_unit"] = df[production_col] / df[downtime_col].replace(0, np.nan)

display(df.head())

In [ ]:
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("Data types:")
display(df.dtypes.to_frame("dtype"))
print("Numeric summary:")
display(df.describe().T)

# Frequency distributions and unique values for categorical columns
categorical_cols = df.select_dtypes(include=["object", "category", "bool", "string"]).columns.tolist()
for col in categorical_cols:
    print(f"\n{col}: unique count = {df[col].nunique(dropna=True)}")
    print("Unique values:", df[col].dropna().unique()[:30])
    display(df[col].value_counts(dropna=False).head(20).to_frame("count"))

# Correlation matrix
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
if len(numeric_cols) >= 2:
    corr_matrix = df[numeric_cols].corr()
    display(corr_matrix)
else:
    print("Not enough numeric columns for a correlation matrix.")

In [ ]:
group_candidates = [c for c in [machine_col, product_col, shift_col] if c]
for group_col in group_candidates:
    metrics = [c for c in numeric_cols if c != group_col]
    if metrics:
        print(f"Average numeric metrics grouped by {group_col}:")
        display(df.groupby(group_col, dropna=False)[metrics].mean().round(2))

if date_col:
    print("Records by date:")
    date_counts = df.groupby(df[date_col].dt.date).size().to_frame("record_count")
    display(date_counts.head(20))

In [ ]:
# Create useful log-transformed numeric fields for skewed production measures
for col in numeric_cols:
    if any(term in col for term in ["production", "output", "quantity", "units", "defect", "downtime", "cost", "energy"]):
        if (df[col].dropna() >= 0).all():
            df["log_" + col] = np.log1p(df[col])

# Group summaries by detected operational dimensions
for group_col in group_candidates:
    measures = [c for c in [production_col, defect_col, downtime_col, target_col,
                            "defect_rate_pct", "target_achievement_pct"] if c and c in df.columns]
    if measures:
        print("Operational summary by", group_col)
        display(df.groupby(group_col, dropna=False)[measures].agg(["count", "mean", "sum"]).round(2))

# Concatenate two parts of the real dataset as a concat demonstration
split_point = len(df) // 2
df_combined = pd.concat([df.iloc[:split_point], df.iloc[split_point:]], ignore_index=True)
print("Original cleaned rows:", len(df), "| Rows after concat:", len(df_combined))
print("Concat retained all rows:", len(df_combined) == len(df))

In [ ]:
stat_columns = df.select_dtypes(include=np.number).columns.tolist()
stat_rows = []
for col in stat_columns:
    values = df[col].dropna()
    modes = values.mode()
    stat_rows.append({
        "metric": col,
        "count": values.count(),
        "mean": values.mean(),
        "median": values.median(),
        "mode": modes.iloc[0] if not modes.empty else np.nan,
        "minimum": values.min(),
        "maximum": values.max(),
        "std_dev": values.std(),
        "variance": values.var(),
        "p25": values.quantile(.25),
        "p50": values.quantile(.50),
        "p75": values.quantile(.75),
        "p90": values.quantile(.90),
        "skewness": values.skew(),
        "kurtosis": values.kurt()
    })
statistics_table = pd.DataFrame(stat_rows).set_index("metric") if stat_rows else pd.DataFrame()
display(statistics_table.round(3))

In [ ]:
if production_col:
    plt.figure(figsize=(8,5))
    plt.hist(df[production_col].dropna(), bins=20, edgecolor="black")
    plt.xlabel(production_col); plt.ylabel("Frequency"); plt.title("Production Quantity Distribution")
    plt.tight_layout(); plt.show()
else: print("Skipped: no production quantity field detected.")

In [ ]:
if date_col and production_col:
    trend = df.dropna(subset=[date_col]).groupby(df[date_col].dt.date)[production_col].sum()
    plt.figure(figsize=(11,5)); plt.plot(pd.to_datetime(trend.index), trend.values, marker="o")
    plt.xlabel("Date"); plt.ylabel("Total production"); plt.title("Production Trend Over Time")
    plt.xticks(rotation=45); plt.tight_layout(); plt.show()
else: print("Skipped: requires date and production fields.")

In [ ]:
bar_group = machine_col or product_col or shift_col
if bar_group and production_col:
    grouped = df.groupby(bar_group)[production_col].mean().sort_values(ascending=False)
    grouped.plot(kind="bar", figsize=(9,5))
    plt.xlabel(bar_group); plt.ylabel("Average production"); plt.title(f"Average Production by {bar_group}")
    plt.xticks(rotation=45); plt.tight_layout(); plt.show()
else: print("Skipped: requires a category field and production field.")

In [ ]:
box_col = production_col or defect_col or downtime_col
if box_col:
    plt.figure(figsize=(8,5)); plt.boxplot(df[box_col].dropna())
    plt.ylabel(box_col); plt.title(f"Distribution and Outliers: {box_col}")
    plt.tight_layout(); plt.show()
else: print("Skipped: no suitable numeric production measure found.")

In [ ]:
numeric_for_plot = df.select_dtypes(include=np.number)
if numeric_for_plot.shape[1] >= 2:
    plt.figure(figsize=(11,8))
    sns.heatmap(numeric_for_plot.corr(), annot=True, fmt=".2f", cmap="coolwarm")
    plt.title("Factory Dataset Correlation Heatmap")
    plt.tight_layout(); plt.show()
else: print("Skipped: requires at least two numeric columns.")

In [ ]:
count_group = machine_col or product_col or shift_col or (categorical_cols[0] if categorical_cols else None)
if count_group:
    plt.figure(figsize=(9,5))
    order = df[count_group].value_counts().index
    sns.countplot(data=df, x=count_group, order=order)
    plt.xlabel(count_group); plt.ylabel("Record count"); plt.title(f"Records by {count_group}")
    plt.xticks(rotation=45); plt.tight_layout(); plt.show()
else: print("Skipped: no categorical column found.")

In [ ]:
if production_col and defect_col:
    plt.figure(figsize=(8,5))
    sns.scatterplot(data=df, x=production_col, y=defect_col)
    plt.title("Production Quantity vs Defects")
    plt.tight_layout(); plt.show()
else: print("Skipped: requires production and defect fields.")

In [ ]:
pair_cols = [c for c in [production_col, defect_col, downtime_col, target_col] if c]
pair_cols = list(dict.fromkeys(pair_cols))
if len(pair_cols) >= 2:
    pair_data = df[pair_cols].dropna()
    if len(pair_data) > 1000:
        pair_data = pair_data.sample(1000, random_state=42)
    sns.pairplot(pair_data, diag_kind="hist")
    plt.show()
else: print("Skipped: need at least two detected production measures.")

In [ ]:
if date_col and production_col:
    plot_df = df.dropna(subset=[date_col]).copy()
    fig = px.line(plot_df, x=date_col, y=production_col, color=shift_col if shift_col else None,
                  title="Interactive Production Trend")
    fig.show()
elif bar_group and production_col:
    plot_df = df.groupby(bar_group, as_index=False)[production_col].mean()
    fig = px.bar(plot_df, x=bar_group, y=production_col, title="Interactive Average Production by Category")
    fig.show()
elif len(numeric_cols) >= 2:
    fig = px.scatter(df, x=numeric_cols[0], y=numeric_cols[1], title="Interactive Numeric Relationship")
    fig.show()
else:
    print("Skipped: insufficient columns for an interactive chart.")

In [ ]:
insights = []

def rank_average(group_col, metric, description):
    if group_col and metric and group_col in df and metric in df:
        result = df.groupby(group_col)[metric].mean().sort_values(ascending=False)
        print(description)
        display(result.to_frame("average_value"))
        if len(result):
            insights.append(f"{description}: {result.index[0]} (average {result.iloc[0]:.2f}).")
        return result
    print(description + ": cannot be calculated because the required fields are unavailable.")
    insights.append(description + ": unavailable in this dataset.")
    return None

best_machine = rank_average(machine_col, production_col, "Highest average production by machine")
best_product = rank_average(product_col, production_col, "Highest average production by product")
best_shift = rank_average(shift_col, production_col, "Highest average production by shift")

if defect_col and (machine_col or product_col or shift_col):
    defect_group = machine_col or product_col or shift_col
    lowest_defects = df.groupby(defect_group)[defect_col].mean().sort_values()
    print("Lowest average defects by", defect_group)
    display(lowest_defects.to_frame("average_defects"))
    if len(lowest_defects):
        insights.append(f"Lowest average defects: {lowest_defects.index[0]} ({lowest_defects.iloc[0]:.2f}).")

if downtime_col and (machine_col or shift_col):
    downtime_group = machine_col or shift_col
    downtime_rank = df.groupby(downtime_group)[downtime_col].mean().sort_values()
    print("Average downtime by", downtime_group)
    display(downtime_rank.to_frame("average_downtime"))
    if len(downtime_rank):
        insights.append(f"Lowest average downtime: {downtime_rank.index[0]} ({downtime_rank.iloc[0]:.2f}).")

if "defect_rate_pct" in df:
    insights.append(f"Overall average defect rate: {df['defect_rate_pct'].mean():.2f}%.")
if "target_achievement_pct" in df:
    insights.append(f"Average target achievement: {df['target_achievement_pct'].mean():.2f}%.")

print("\nInsight notes:")
for item in insights:
    print("-", item)

In [ ]:
summary_lines = [
    "FACTORY PRODUCTION ANALYTICS — ONE-PAGE SUMMARY",
    f"Rows analysed after duplicate removal: {len(df)}",
    f"Columns in dataset: {len(df.columns)}",
    "",
    "DATA QUALITY",
    f"Remaining missing values: {int(df.isna().sum().sum())}",
    f"Duplicate rows removed: {int(df.duplicated().sum())}",
    "",
    "KEY OPERATIONAL INSIGHTS"
] + ["- " + item for item in insights]

if production_col:
    summary_lines += [
        "",
        "PRODUCTION STATISTICS",
        f"Average {production_col}: {df[production_col].mean():.2f}",
        f"Median {production_col}: {df[production_col].median():.2f}",
        f"Minimum {production_col}: {df[production_col].min():.2f}",
        f"Maximum {production_col}: {df[production_col].max():.2f}"
    ]

summary_lines += [
    "",
    "RECOMMENDATIONS",
    "1. Compare production output across the available machines, products, or shifts.",
    "2. Investigate groups with higher defect counts or downtime, if those measures are present.",
    "3. Use the correlation matrix to identify measures that move together; correlation does not prove causation.",
    "4. Review missing values and unusual observations before making operational decisions.",
    "",
    "LIMITATION",
    "Conclusions are descriptive and depend on the fields and records contained in the supplied workbook."
]
summary_text = "\n".join(summary_lines)
print(summary_text)
Path("factory_production_one_page_summary.txt").write_text(summary_text, encoding="utf-8")
print("\nSaved: factory_production_one_page_summary.txt")

In [ ]:
df.to_excel("factory_production_cleaned.xlsx", index=False)
df.to_csv("factory_production_cleaned.csv", index=False)
print("Saved factory_production_cleaned.xlsx and factory_production_cleaned.csv")